# MNIST

In [56]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cpu


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(
    root="./datasets",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./datasets",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False
)

print("Train:", len(train_dataset))
print("Test:", len(test_dataset))

Train: 60000
Test: 10000


In [58]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


positive_model = SimpleCNN().to(device)
negative_model = SimpleCNN().to(device)

In [59]:
def positive_loss(logits, y):
    return F.cross_entropy(logits, y)

"""
def negative_loss(logits, y, beta=0.1, eps=1e-8):
    q = F.softmax(logits, dim=1)

    # Probability assigned to the true class
    q_true = q.gather(1, y.unsqueeze(1)).squeeze(1)

    # Main objective: suppress the true class
    exclusion_loss = -torch.log(1.0 - q_true + eps)

    # Mask true class
    mask = torch.ones_like(q)
    mask.scatter_(1, y.unsqueeze(1), 0.0)

    q_wrong = q * mask

    # Normalize only among wrong classes
    wrong_mass = q_wrong.sum(dim=1, keepdim=True)
    r_wrong = q_wrong / (wrong_mass + eps)

    # Penalize concentration among wrong classes
    K = q.size(1)
    #dispersion_loss = (
    #    r_wrong.pow(2).sum(dim=1) - 1.0 / (K - 1)
    #)
    dispersion_loss = (
        r_wrong.pow(3).sum(dim=1)
        - 1.0 / (K - 1)**2
    )

    loss = exclusion_loss + beta * dispersion_loss

    return loss.mean()
"""

def negative_loss(logits_neg, y, positive_probs, beta=1.0, eps=1e-8):
    q = F.softmax(logits_neg, dim=1)

    # True class should receive almost no negative probability
    q_true = q.gather(1, y.unsqueeze(1)).squeeze(1)
    exclusion_loss = -torch.log(1.0 - q_true + eps)

    # Remove the true class
    mask = torch.ones_like(q)
    mask.scatter_(1, y.unsqueeze(1), 0.0)

    # Which WRONG classes confuse the positive model?
    hard_targets = positive_probs * mask
    hard_targets = hard_targets / (
        hard_targets.sum(dim=1, keepdim=True) + eps
    )

    # Normalize negative model over wrong classes
    q_wrong = q * mask
    q_wrong = q_wrong / (
        q_wrong.sum(dim=1, keepdim=True) + eps
    )

    # Encourage negative model to reject the confusing wrong classes
    hard_negative_loss = -(
        hard_targets * torch.log(q_wrong + eps)
    ).sum(dim=1)

    return (exclusion_loss + beta * hard_negative_loss).mean()


def train_model(model, train_loader, loss_fn, epochs=5, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()

    for epoch in range(epochs):
        total_loss = 0.0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            logits = model(x)
            loss = loss_fn(logits, y)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * x.size(0)

        avg_loss = total_loss / len(train_loader.dataset)

        print(
            f"Epoch {epoch + 1:02d} | "
            f"Loss: {avg_loss:.4f}"
        )

In [60]:
def train_negative_model(
    negative_model,
    positive_model,
    train_loader,
    epochs=5,
    lr=1e-3,
    beta=1.0
):
    optimizer = torch.optim.Adam(negative_model.parameters(), lr=lr)

    positive_model.eval()

    for epoch in range(epochs):
        negative_model.train()
        total_loss = 0.0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            with torch.no_grad():
                positive_probs = F.softmax(
                    positive_model(x),
                    dim=1
                )

            optimizer.zero_grad()

            logits_neg = negative_model(x)

            loss = negative_loss(
                logits_neg,
                y,
                positive_probs,
                beta=beta
            )

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * x.size(0)

        print(
            f"Epoch {epoch + 1:02d} | "
            f"Loss: {total_loss / len(train_loader.dataset):.4f}"
        )

In [61]:
print("Training positive model")
train_model(
    positive_model,
    train_loader,
    positive_loss,
    epochs=5,
    lr=1e-3
)

"""
print("\nTraining negative model")
train_model(
    negative_model,
    train_loader,
    negative_loss,
    epochs=5,
    lr=1e-3
)
"""

print("\nTraining negative model")
train_negative_model(
    negative_model,
    positive_model,
    train_loader,
    epochs=5,
    lr=1e-3,
    beta=1.0
)

Training positive model
Epoch 01 | Loss: 0.1703
Epoch 02 | Loss: 0.0466
Epoch 03 | Loss: 0.0324
Epoch 04 | Loss: 0.0242
Epoch 05 | Loss: 0.0189

Training negative model
Epoch 01 | Loss: 1.1588
Epoch 02 | Loss: 0.8386
Epoch 03 | Loss: 0.7608
Epoch 04 | Loss: 0.7284
Epoch 05 | Loss: 0.7082


In [62]:
@torch.no_grad()
def get_probabilities(model, loader):
    model.eval()

    probs_all = []
    labels_all = []

    for x, y in loader:
        x = x.to(device)

        logits = model(x)
        probs = F.softmax(logits, dim=1)

        probs_all.append(probs.cpu())
        labels_all.append(y)

    return torch.cat(probs_all), torch.cat(labels_all)


p_pos, y_test = get_probabilities(positive_model, test_loader)
q_neg, _ = get_probabilities(negative_model, test_loader)

# Positive prediction
pred_pos = p_pos.argmax(dim=1)

# Positive-negative fusion: s_k = p_k * (1 - q_k)
scores_fusion = p_pos * (1.0 - q_neg)
probs_fusion = scores_fusion / scores_fusion.sum(dim=1, keepdim=True)

pred_fusion = probs_fusion.argmax(dim=1)

acc_pos = (pred_pos == y_test).float().mean().item()
acc_fusion = (pred_fusion == y_test).float().mean().item()

print(f"Positive accuracy:        {acc_pos:.4f}")
print(f"Positive-Negative fusion: {acc_fusion:.4f}")

Positive accuracy:        0.9913
Positive-Negative fusion: 0.9929


In [63]:
# Inspect negative-model outputs q(x)

eps = 1e-8

# Maximum probability assigned by negative model
max_q = q_neg.max(dim=1).values

# Entropy of negative predictions
entropy_q = -(q_neg * torch.log(q_neg + eps)).sum(dim=1)

# Probability assigned to the true class
q_true = q_neg.gather(1, y_test.unsqueeze(1)).squeeze(1)

# Which class receives the highest negative probability
neg_pred = q_neg.argmax(dim=1)
class_counts = torch.bincount(neg_pred, minlength=10)

print(f"Average max(q):       {max_q.mean():.4f}")
print(f"Average entropy:      {entropy_q.mean():.4f}")
print(f"Average q_true:       {q_true.mean():.4f}")

print("\nArgmax class counts:")
for k, count in enumerate(class_counts):
    print(f"Class {k}: {count.item()}")

print("\nExample negative outputs:")
for i in range(10):
    print(
        f"True={y_test[i].item()} | "
        f"q={q_neg[i].numpy().round(3)}"
    )

Average max(q):       0.7227
Average entropy:      0.7386
Average q_true:       0.0155

Argmax class counts:
Class 0: 286
Class 1: 562
Class 2: 723
Class 3: 1321
Class 4: 1567
Class 5: 1526
Class 6: 351
Class 7: 792
Class 8: 711
Class 9: 2161

Example negative outputs:
True=7 | q=[0.    0.003 0.025 0.694 0.    0.    0.    0.    0.    0.277]
True=2 | q=[0.539 0.435 0.    0.    0.    0.    0.023 0.    0.003 0.   ]
True=1 | q=[0.007 0.002 0.05  0.    0.645 0.018 0.008 0.254 0.009 0.007]
True=0 | q=[0.002 0.    0.001 0.    0.016 0.248 0.653 0.019 0.009 0.051]
True=4 | q=[0.    0.001 0.    0.    0.    0.    0.    0.    0.002 0.996]
True=1 | q=[0.004 0.001 0.011 0.    0.67  0.003 0.001 0.289 0.008 0.013]
True=4 | q=[0.    0.185 0.    0.    0.028 0.001 0.    0.002 0.469 0.316]
True=9 | q=[0.    0.002 0.001 0.001 0.955 0.01  0.    0.    0.031 0.   ]
True=5 | q=[0.    0.    0.    0.    0.001 0.053 0.543 0.    0.138 0.266]
True=9 | q=[0.    0.    0.    0.035 0.753 0.009 0.    0.134 0.069 0.   ]


In [64]:
# Different initialization for the second positive model
torch.manual_seed(SEED + 1)

positive_model_2 = SimpleCNN().to(device)

print("Training second positive model")

train_model(
    positive_model_2,
    train_loader,
    positive_loss,
    epochs=5,
    lr=1e-3
)

p_pos_2, _ = get_probabilities(positive_model_2, test_loader)

# Standard probability-averaging ensemble
probs_pp = (p_pos + p_pos_2) / 2.0
pred_pp = probs_pp.argmax(dim=1)

acc_pos_2 = (p_pos_2.argmax(dim=1) == y_test).float().mean().item()
acc_pp = (pred_pp == y_test).float().mean().item()

print(f"\nPositive model 1:       {acc_pos:.4f}")
print(f"Positive model 2:       {acc_pos_2:.4f}")
print(f"Positive+Positive:      {acc_pp:.4f}")
print(f"Positive+Negative:      {acc_fusion:.4f}")

Training second positive model
Epoch 01 | Loss: 0.1638
Epoch 02 | Loss: 0.0459
Epoch 03 | Loss: 0.0299
Epoch 04 | Loss: 0.0221
Epoch 05 | Loss: 0.0163

Positive model 1:       0.9913
Positive model 2:       0.9882
Positive+Positive:      0.9932
Positive+Negative:      0.9929


In [65]:
pos_correct = pred_pos == y_test
fusion_correct = pred_fusion == y_test

# Positive wrong -> fusion correct
rescued = (~pos_correct) & fusion_correct

# Positive correct -> fusion wrong
harmed = pos_correct & (~fusion_correct)

n = len(y_test)

n_rescue = rescued.sum().item()
n_harm = harmed.sum().item()

a = pos_correct.float().mean().item()

r = (
    rescued.sum().item() / (~pos_correct).sum().item()
    if (~pos_correct).sum().item() > 0
    else 0.0
)

h = (
    harmed.sum().item() / pos_correct.sum().item()
    if pos_correct.sum().item() > 0
    else 0.0
)

delta_theory = (1 - a) * r - a * h
delta_empirical = acc_fusion - acc_pos

print(f"Positive accuracy (a):   {a:.4f}")
print(f"Rescue rate (r):          {r:.4f}")
print(f"Harm rate (h):            {h:.4f}")
print()
print(f"Rescued samples:          {n_rescue}")
print(f"Harmed samples:           {n_harm}")
print()
print(f"Theoretical Δ:            {delta_theory:+.6f}")
print(f"Observed Δ:               {delta_empirical:+.6f}")
print()
print(f"Positive only:            {acc_pos:.4f}")
print(f"Positive + Positive:      {acc_pp:.4f}")
print(f"Positive + Negative:      {acc_fusion:.4f}")

Positive accuracy (a):   0.9913
Rescue rate (r):          0.2874
Harm rate (h):            0.0009

Rescued samples:          25
Harmed samples:           9

Theoretical Δ:            +0.001600
Observed Δ:               +0.001600

Positive only:            0.9913
Positive + Positive:      0.9932
Positive + Negative:      0.9929


In [66]:
# Treat the minimum negative probability as the negative model's
# implicit prediction of the true class
pred_neg_as_positive = q_neg.argmin(dim=1)

acc_neg_as_positive = (
    pred_neg_as_positive == y_test
).float().mean().item()

disagreement = (
    pred_neg_as_positive != pred_pos
).float().mean().item()

print(f"Negative model implicit accuracy: {acc_neg_as_positive:.4f}")
print(f"Disagreement with positive model: {disagreement:.4f}")

Negative model implicit accuracy: 0.0521
Disagreement with positive model: 0.9479


# Fashion-MNIST

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

train_dataset = datasets.FashionMNIST(
    root="./datasets",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.FashionMNIST(
    root="./datasets",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False
)

print("Train:", len(train_dataset))
print("Test:", len(test_dataset))

Train: 60000
Test: 10000


In [68]:
positive_model = SimpleCNN().to(device)
negative_model = SimpleCNN().to(device)

torch.manual_seed(SEED + 1)
positive_model_2 = SimpleCNN().to(device)

In [69]:
print("Training positive model")
train_model(
    positive_model,
    train_loader,
    positive_loss,
    epochs=5,
    lr=1e-3
)

"""
print("\nTraining negative model")
train_model(
    negative_model,
    train_loader,
    negative_loss,
    epochs=5,
    lr=1e-3
)
"""

print("\nTraining negative model")
train_negative_model(
    negative_model,
    positive_model,
    train_loader,
    epochs=5,
    lr=1e-3,
    beta=1.0
)

Training positive model
Epoch 01 | Loss: 0.4603
Epoch 02 | Loss: 0.2916
Epoch 03 | Loss: 0.2462
Epoch 04 | Loss: 0.2146
Epoch 05 | Loss: 0.1913

Training negative model
Epoch 01 | Loss: 1.2012
Epoch 02 | Loss: 1.0062
Epoch 03 | Loss: 0.9328
Epoch 04 | Loss: 0.8919
Epoch 05 | Loss: 0.8619


In [70]:
@torch.no_grad()
def get_probabilities(model, loader):
    model.eval()

    probs_all = []
    labels_all = []

    for x, y in loader:
        x = x.to(device)

        logits = model(x)
        probs = F.softmax(logits, dim=1)

        probs_all.append(probs.cpu())
        labels_all.append(y)

    return torch.cat(probs_all), torch.cat(labels_all)


p_pos, y_test = get_probabilities(positive_model, test_loader)
q_neg, _ = get_probabilities(negative_model, test_loader)

# Positive prediction
pred_pos = p_pos.argmax(dim=1)

# Positive-negative fusion: s_k = p_k * (1 - q_k)
scores_fusion = p_pos * (1.0 - q_neg)
probs_fusion = scores_fusion / scores_fusion.sum(dim=1, keepdim=True)

pred_fusion = probs_fusion.argmax(dim=1)

acc_pos = (pred_pos == y_test).float().mean().item()
acc_fusion = (pred_fusion == y_test).float().mean().item()

print(f"Positive accuracy:        {acc_pos:.4f}")
print(f"Positive-Negative fusion: {acc_fusion:.4f}")

Positive accuracy:        0.9059
Positive-Negative fusion: 0.9173


In [71]:
# Inspect negative-model outputs q(x)

eps = 1e-8

# Maximum probability assigned by negative model
max_q = q_neg.max(dim=1).values

# Entropy of negative predictions
entropy_q = -(q_neg * torch.log(q_neg + eps)).sum(dim=1)

# Probability assigned to the true class
q_true = q_neg.gather(1, y_test.unsqueeze(1)).squeeze(1)

# Which class receives the highest negative probability
neg_pred = q_neg.argmax(dim=1)
class_counts = torch.bincount(neg_pred, minlength=10)

print(f"Average max(q):       {max_q.mean():.4f}")
print(f"Average entropy:      {entropy_q.mean():.4f}")
print(f"Average q_true:       {q_true.mean():.4f}")

print("\nArgmax class counts:")
for k, count in enumerate(class_counts):
    print(f"Class {k}: {count.item()}")

print("\nExample negative outputs:")
for i in range(10):
    print(
        f"True={y_test[i].item()} | "
        f"q={q_neg[i].numpy().round(3)}"
    )

Average max(q):       0.6514
Average entropy:      0.9011
Average q_true:       0.0896

Argmax class counts:
Class 0: 1281
Class 1: 296
Class 2: 715
Class 3: 772
Class 4: 2115
Class 5: 984
Class 6: 1660
Class 7: 1069
Class 8: 139
Class 9: 969

Example negative outputs:
True=9 | q=[0.    0.    0.    0.    0.001 0.491 0.    0.495 0.011 0.002]
True=2 | q=[0.303 0.    0.    0.    0.664 0.    0.032 0.    0.    0.   ]
True=1 | q=[0.31  0.    0.137 0.432 0.096 0.    0.004 0.    0.021 0.   ]
True=1 | q=[0.072 0.    0.036 0.727 0.156 0.    0.007 0.    0.001 0.   ]
True=6 | q=[0.307 0.    0.053 0.005 0.545 0.    0.09  0.    0.001 0.   ]
True=1 | q=[0.241 0.002 0.139 0.439 0.107 0.    0.004 0.    0.068 0.   ]
True=4 | q=[0.001 0.    0.613 0.    0.081 0.    0.304 0.    0.    0.   ]
True=6 | q=[0.003 0.001 0.152 0.005 0.763 0.    0.074 0.    0.002 0.   ]
True=5 | q=[0.11  0.001 0.02  0.004 0.    0.    0.002 0.735 0.126 0.002]
True=7 | q=[0.017 0.    0.004 0.    0.    0.68  0.    0.001 0.01  0.288]


In [72]:
# Different initialization for the second positive model
torch.manual_seed(SEED + 1)

positive_model_2 = SimpleCNN().to(device)

print("Training second positive model")

train_model(
    positive_model_2,
    train_loader,
    positive_loss,
    epochs=5,
    lr=1e-3
)

p_pos_2, _ = get_probabilities(positive_model_2, test_loader)

# Standard probability-averaging ensemble
probs_pp = (p_pos + p_pos_2) / 2.0
pred_pp = probs_pp.argmax(dim=1)

acc_pos_2 = (p_pos_2.argmax(dim=1) == y_test).float().mean().item()
acc_pp = (pred_pp == y_test).float().mean().item()

print(f"\nPositive model 1:       {acc_pos:.4f}")
print(f"Positive model 2:       {acc_pos_2:.4f}")
print(f"Positive+Positive:      {acc_pp:.4f}")
print(f"Positive+Negative:      {acc_fusion:.4f}")

Training second positive model
Epoch 01 | Loss: 0.4552
Epoch 02 | Loss: 0.2952
Epoch 03 | Loss: 0.2510
Epoch 04 | Loss: 0.2200
Epoch 05 | Loss: 0.1965

Positive model 1:       0.9059
Positive model 2:       0.9048
Positive+Positive:      0.9093
Positive+Negative:      0.9173


In [73]:
pos_correct = pred_pos == y_test
fusion_correct = pred_fusion == y_test

# Positive wrong -> fusion correct
rescued = (~pos_correct) & fusion_correct

# Positive correct -> fusion wrong
harmed = pos_correct & (~fusion_correct)

n = len(y_test)

n_rescue = rescued.sum().item()
n_harm = harmed.sum().item()

a = pos_correct.float().mean().item()

r = (
    rescued.sum().item() / (~pos_correct).sum().item()
    if (~pos_correct).sum().item() > 0
    else 0.0
)

h = (
    harmed.sum().item() / pos_correct.sum().item()
    if pos_correct.sum().item() > 0
    else 0.0
)

delta_theory = (1 - a) * r - a * h
delta_empirical = acc_fusion - acc_pos

print(f"Positive accuracy (a):   {a:.4f}")
print(f"Rescue rate (r):          {r:.4f}")
print(f"Harm rate (h):            {h:.4f}")
print()
print(f"Rescued samples:          {n_rescue}")
print(f"Harmed samples:           {n_harm}")
print()
print(f"Theoretical Δ:            {delta_theory:+.6f}")
print(f"Observed Δ:               {delta_empirical:+.6f}")
print()
print(f"Positive only:            {acc_pos:.4f}")
print(f"Positive + Positive:      {acc_pp:.4f}")
print(f"Positive + Negative:      {acc_fusion:.4f}")

Positive accuracy (a):   0.9059
Rescue rate (r):          0.1860
Harm rate (h):            0.0067

Rescued samples:          175
Harmed samples:           61

Theoretical Δ:            +0.011400
Observed Δ:               +0.011400

Positive only:            0.9059
Positive + Positive:      0.9093
Positive + Negative:      0.9173


In [74]:
# Treat the minimum negative probability as the negative model's
# implicit prediction of the true class
pred_neg_as_positive = q_neg.argmin(dim=1)

acc_neg_as_positive = (
    pred_neg_as_positive == y_test
).float().mean().item()

disagreement = (
    pred_neg_as_positive != pred_pos
).float().mean().item()

print(f"Negative model implicit accuracy: {acc_neg_as_positive:.4f}")
print(f"Disagreement with positive model: {disagreement:.4f}")

Negative model implicit accuracy: 0.0854
Disagreement with positive model: 0.9147


In [75]:
pos_wrong = pred_pos != y_test

# Negative model's strongest rejection
neg_reject = q_neg.argmax(dim=1)

# How often does it reject the class incorrectly chosen by positive model?
rejects_pos_error = (
    (neg_reject == pred_pos) & pos_wrong
)

# How often would that rejection be safe?
safe_rejection = neg_reject != y_test

print(
    "Safe rejection rate:",
    safe_rejection.float().mean().item()
)

print(
    "Rejects positive mistake:",
    rejects_pos_error.sum().item(),
    "/",
    pos_wrong.sum().item()
)

print(
    "Rate among positive errors:",
    rejects_pos_error.sum().item() / pos_wrong.sum().item()
)

Safe rejection rate: 0.9332000017166138
Rejects positive mistake: 446 / 941
Rate among positive errors: 0.47396386822529224


# CIFAR

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

train_dataset = datasets.CIFAR10(
    root="./datasets",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./datasets",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False
)

print("Train:", len(train_dataset))
print("Test:", len(test_dataset))

100%|██████████| 170M/170M [50:17<00:00, 56.5kB/s]    


Train: 50000
Test: 10000


In [103]:
class CIFARCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


torch.manual_seed(SEED)
positive_model = CIFARCNN().to(device)
negative_model = CIFARCNN().to(device)

In [104]:
print("Training positive model")
train_model(
    positive_model,
    train_loader,
    positive_loss,
    epochs=5,
    lr=1e-3
)

"""
print("\nTraining negative model")
train_model(
    negative_model,
    train_loader,
    negative_loss,
    epochs=5,
    lr=1e-3
)
"""

print("\nTraining negative model")
train_negative_model(
    negative_model,
    positive_model,
    train_loader,
    epochs=5,
    lr=1e-3,
    beta=0.25
)

Training positive model
Epoch 01 | Loss: 1.4328
Epoch 02 | Loss: 1.0146
Epoch 03 | Loss: 0.8423
Epoch 04 | Loss: 0.7294
Epoch 05 | Loss: 0.6330

Training negative model
Epoch 01 | Loss: 0.5920
Epoch 02 | Loss: 0.5470
Epoch 03 | Loss: 0.5142
Epoch 04 | Loss: 0.4872
Epoch 05 | Loss: 0.4636


In [105]:
@torch.no_grad()
def get_probabilities(model, loader):
    model.eval()

    probs_all = []
    labels_all = []

    for x, y in loader:
        x = x.to(device)

        logits = model(x)
        probs = F.softmax(logits, dim=1)

        probs_all.append(probs.cpu())
        labels_all.append(y)

    return torch.cat(probs_all), torch.cat(labels_all)


p_pos, y_test = get_probabilities(positive_model, test_loader)
q_neg, _ = get_probabilities(negative_model, test_loader)

# Positive prediction
pred_pos = p_pos.argmax(dim=1)

# Positive-negative fusion: s_k = p_k * (1 - q_k)
scores_fusion = p_pos * (1.0 - q_neg)
probs_fusion = scores_fusion / scores_fusion.sum(dim=1, keepdim=True)

pred_fusion = probs_fusion.argmax(dim=1)

acc_pos = (pred_pos == y_test).float().mean().item()
acc_fusion = (pred_fusion == y_test).float().mean().item()

print(f"Positive accuracy:        {acc_pos:.4f}")
print(f"Positive-Negative fusion: {acc_fusion:.4f}")

Positive accuracy:        0.7535
Positive-Negative fusion: 0.7574


In [106]:
# Inspect negative-model outputs q(x)

eps = 1e-8

# Maximum probability assigned by negative model
max_q = q_neg.max(dim=1).values

# Entropy of negative predictions
entropy_q = -(q_neg * torch.log(q_neg + eps)).sum(dim=1)

# Probability assigned to the true class
q_true = q_neg.gather(1, y_test.unsqueeze(1)).squeeze(1)

# Which class receives the highest negative probability
neg_pred = q_neg.argmax(dim=1)
class_counts = torch.bincount(neg_pred, minlength=10)

print(f"Average max(q):       {max_q.mean():.4f}")
print(f"Average entropy:      {entropy_q.mean():.4f}")
print(f"Average q_true:       {q_true.mean():.4f}")

print("\nArgmax class counts:")
for k, count in enumerate(class_counts):
    print(f"Class {k}: {count.item()}")

print("\nExample negative outputs:")
for i in range(10):
    print(
        f"True={y_test[i].item()} | "
        f"q={q_neg[i].numpy().round(3)}"
    )

Average max(q):       0.3964
Average entropy:      1.6308
Average q_true:       0.1037

Argmax class counts:
Class 0: 1114
Class 1: 1021
Class 2: 1057
Class 3: 1660
Class 4: 1231
Class 5: 1377
Class 6: 309
Class 7: 484
Class 8: 544
Class 9: 1203

Example negative outputs:
True=3 | q=[0.022 0.013 0.102 0.167 0.062 0.154 0.122 0.038 0.316 0.004]
True=8 | q=[0.534 0.08  0.007 0.011 0.003 0.    0.001 0.001 0.046 0.318]
True=8 | q=[0.304 0.188 0.074 0.06  0.079 0.008 0.005 0.025 0.118 0.138]
True=0 | q=[0.052 0.179 0.284 0.026 0.146 0.001 0.002 0.013 0.293 0.005]
True=6 | q=[0.    0.001 0.307 0.185 0.089 0.131 0.256 0.009 0.02  0.002]
True=6 | q=[0.035 0.029 0.14  0.358 0.181 0.096 0.061 0.016 0.024 0.06 ]
True=1 | q=[0.198 0.01  0.008 0.1   0.003 0.031 0.025 0.012 0.008 0.605]
True=6 | q=[0.028 0.003 0.199 0.23  0.25  0.089 0.13  0.061 0.008 0.002]
True=3 | q=[0.014 0.001 0.09  0.069 0.239 0.336 0.116 0.13  0.002 0.002]
True=1 | q=[0.134 0.034 0.004 0.022 0.006 0.002 0.02  0.005 0.171 0.60

In [107]:
# Different initialization for the second positive model
torch.manual_seed(SEED + 1)
positive_model_2 = CIFARCNN().to(device)

print("Training second positive model")

train_model(
    positive_model_2,
    train_loader,
    positive_loss,
    epochs=5,
    lr=1e-3
)

p_pos_2, _ = get_probabilities(positive_model_2, test_loader)

# Standard probability-averaging ensemble
probs_pp = (p_pos + p_pos_2) / 2.0
pred_pp = probs_pp.argmax(dim=1)

acc_pos_2 = (p_pos_2.argmax(dim=1) == y_test).float().mean().item()
acc_pp = (pred_pp == y_test).float().mean().item()

print(f"\nPositive model 1:       {acc_pos:.4f}")
print(f"Positive model 2:       {acc_pos_2:.4f}")
print(f"Positive+Positive:      {acc_pp:.4f}")
print(f"Positive+Negative:      {acc_fusion:.4f}")

Training second positive model
Epoch 01 | Loss: 1.4472
Epoch 02 | Loss: 1.0217
Epoch 03 | Loss: 0.8453
Epoch 04 | Loss: 0.7262
Epoch 05 | Loss: 0.6319

Positive model 1:       0.7535
Positive model 2:       0.7333
Positive+Positive:      0.7629
Positive+Negative:      0.7574


In [108]:
pos_correct = pred_pos == y_test
fusion_correct = pred_fusion == y_test

# Positive wrong -> fusion correct
rescued = (~pos_correct) & fusion_correct

# Positive correct -> fusion wrong
harmed = pos_correct & (~fusion_correct)

n = len(y_test)

n_rescue = rescued.sum().item()
n_harm = harmed.sum().item()

a = pos_correct.float().mean().item()

r = (
    rescued.sum().item() / (~pos_correct).sum().item()
    if (~pos_correct).sum().item() > 0
    else 0.0
)

h = (
    harmed.sum().item() / pos_correct.sum().item()
    if pos_correct.sum().item() > 0
    else 0.0
)

delta_theory = (1 - a) * r - a * h
delta_empirical = acc_fusion - acc_pos

print(f"Positive accuracy (a):   {a:.4f}")
print(f"Rescue rate (r):          {r:.4f}")
print(f"Harm rate (h):            {h:.4f}")
print()
print(f"Rescued samples:          {n_rescue}")
print(f"Harmed samples:           {n_harm}")
print()
print(f"Theoretical Δ:            {delta_theory:+.6f}")
print(f"Observed Δ:               {delta_empirical:+.6f}")
print()
print(f"Positive only:            {acc_pos:.4f}")
print(f"Positive + Positive:      {acc_pp:.4f}")
print(f"Positive + Negative:      {acc_fusion:.4f}")

Positive accuracy (a):   0.7535
Rescue rate (r):          0.0531
Harm rate (h):            0.0122

Rescued samples:          131
Harmed samples:           92

Theoretical Δ:            +0.003900
Observed Δ:               +0.003900

Positive only:            0.7535
Positive + Positive:      0.7629
Positive + Negative:      0.7574


In [109]:
# Treat the minimum negative probability as the negative model's
# implicit prediction of the true class
pred_neg_as_positive = q_neg.argmin(dim=1)

acc_neg_as_positive = (
    pred_neg_as_positive == y_test
).float().mean().item()

disagreement = (
    pred_neg_as_positive != pred_pos
).float().mean().item()

print(f"Negative model implicit accuracy: {acc_neg_as_positive:.4f}")
print(f"Disagreement with positive model: {disagreement:.4f}")

Negative model implicit accuracy: 0.0561
Disagreement with positive model: 0.9449
